# Week 1 Day 1 — Extra Exercise: Playwright Web Scraper

**Author:** slavapa

The standard `scraper.py` uses `requests` + BeautifulSoup, which fails on JavaScript-rendered sites (e.g. OpenAI.com, Kabbalah Media).

This notebook uses a **Playwright fallback** scraper and summarizes three exercise sites:

- https://kabbalahmedia.info/en/
- https://www.michaellaitman.com/
- https://kabuconnect.com/

In [ ]:
import os
import sys

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from scraper_playwright import fetch_website_contents

load_dotenv(override=True)
openai = OpenAI()

## Compare: basic requests vs Playwright

Kabbalah Media is a JavaScript-rendered site — the basic scraper returns almost nothing.

In [ ]:
import sys
sys.path.insert(0, "../../")
from scraper import fetch_website_contents as fetch_basic

url = "https://kabbalahmedia.info/en/"
basic = fetch_basic(url)
playwright = fetch_website_contents(url)

print(f"Basic scraper:      {len(basic)} chars")
print(basic[:200])
print()
print(f"Playwright scraper: {len(playwright)} chars")
print(playwright[:500])

## Summarize all three exercise sites

In [ ]:
EXERCISE_SITES = [
    "https://kabbalahmedia.info/en/",
    "https://www.michaellaitman.com/",
    "https://kabuconnect.com/",
]

system_prompt = """
You are a helpful assistant that analyzes website contents and provides a concise summary.
Ignore navigation menus, footers, and cookie banners.
Respond in markdown. Do not wrap the markdown in a code block.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""


def messages_for(website: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]


def summarize(url: str) -> str:
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website),
    )
    return response.choices[0].message.content


def display_summary(url: str) -> None:
    print(f"### {url}\n")
    display(Markdown(summarize(url)))

In [ ]:
for site in EXERCISE_SITES:
    display_summary(site)
    print("\n---\n")